# Capítulo 7 — Desempenho de Arquiteturas Cliente-Servidor

**Referência:** Capítulo 7 do livro-texto  
**Biblioteca:** `line_solver` — `pfqn_mvald` (MVA exato para redes com taxas dependentes de carga)


---
### Exercício 7.1 — Modelo CS com servidor de 2 processadores

Considere o modelo CS discutido no problema da empresa de telemarketing e assuma que o servidor será atualizado por um servidor de 2 processadores onde cada processador é idêntico ao processador original. Se o mesmo disco é utilizado, resolva o modelo CS assumindo que a taxa de serviço μ(j) para a CPU do servidor é tal que **μ(2) = 1,8·μ(1)**.

**Parâmetros:**
- N\_SQL = 4 requisições SQL por transação
- D\_CL = 45 s (tempo de reflexão do cliente)
- D\_CPU = 0,12 s (demanda CPU por req SQL)
- D\_DSK = 0,054 s (demanda disco por req SQL)
- LAN Ethernet: B = 10 Mbps, S\_slot = 51,2 µs, L\_p = 1518 bits, NP = 7 pacotes/req
- 30.000 chamadas/dia, jornada de 8 h

**Objetivo:** encontrar o número mínimo N de representantes tal que W\_q ≤ 5 s.


In [ ]:
import warnings
warnings.filterwarnings("ignore", category=SyntaxWarning)

import numpy as np
import matplotlib.pyplot as plt
from scipy.special import gammaln
from line_solver.api.pfqn import pfqn_mvald

# ── Parâmetros ─────────────────────────────────────────────
N_SQL      = 4           # requisições SQL por transação
D_CL       = 45.0        # tempo de reflexão do cliente (s)
D_CPU      = 0.12        # demanda CPU por req SQL (s)
D_DSK      = 0.054       # demanda disco por req SQL (s)
B          = 10e6        # largura de banda Ethernet (bps)
S_SLOT     = 51.2e-6     # tempo de slot Ethernet (s)
L_P        = 1518.0      # comprimento médio do pacote (bits)
NP         = 7           # pacotes por requisição SQL

CALLS_DAY  = 30_000
HOURS_DAY  = 8
lam_calls  = CALLS_DAY / (HOURS_DAY * 3600)   # chamadas/s

# ── Parâmetros derivados da LAN ────────────────────────────
t_p   = L_P / B                  # tempo de transmissão (s)
a     = S_SLOT / t_p             # razão de normalização
D_LAN = NP * t_p                 # demanda LAN por req SQL (s)

print(f"lam_calls = {lam_calls:.6f} chamadas/s")
print(f"t_p       = {t_p*1e6:.2f} µs")
print(f"a         = {a:.6f}")
print(f"D_LAN     = {D_LAN*1e3:.4f} ms")
print(f"Limite de estabilidade (N > lam*N_SQL*D_CL): {lam_calls*N_SQL*D_CL:.1f}")


In [ ]:
def build_mu(n, two_cpu=True):
    """Constrói matriz mu (3 x n) para LAN/CPU/Disco."""
    mu = np.ones((3, n))
    # Estação 0: LAN Ethernet  η(k) = 1 / (1 + 2a(k-1))
    for k in range(1, n + 1):
        mu[0, k - 1] = 1.0 / (1.0 + 2.0 * a * (k - 1))
    # Estação 1: CPU  μ(1)=1, μ(k)=1.8 para k≥2 (dois processadores)
    if two_cpu and n > 1:
        mu[1, 1:] = 1.8
    # Estação 2: Disco FCFS → mu=1 (já preenchido)
    return mu


def solve_cs(n, two_cpu=True):
    """Resolve o modelo CS com pfqn_mvald. Retorna (X_sql, R_vec, U_vec)."""
    L   = np.array([[D_LAN], [D_CPU], [D_DSK]])   # demandas (3×1)
    Z_a = np.array([float(D_CL)])                  # pensa times (1,)
    mu  = build_mu(int(n), two_cpu=two_cpu)
    XN, QN, UN, CN, *_ = pfqn_mvald(L, np.array([float(n)]), Z_a, mu)
    X_sql = float(np.ravel(XN)[0])
    R_vec = QN[:, 0] / X_sql        # tempo de resposta por estação
    U_vec = UN[:, 0]                 # utilização por estação
    return X_sql, R_vec, U_vec


# Verificação: tabela η(k)
print(f"{'k':>6}  {'η(k)':>10}")
print("-" * 20)
for k in [1, 5, 10, 20, 50, 100, 200, 250]:
    print(f"{k:>6}  {1/(1+2*a*(k-1)):>10.6f}")


In [ ]:
def erlang_c(c, rho):
    """Erlang-C P(C) — estável para c grande via espaço log."""
    if rho >= 1.0:
        return 1.0
    a_load = c * rho
    k_arr = np.arange(c, dtype=float)
    log_terms  = k_arr * np.log(a_load) - gammaln(k_arr + 1)
    log_erlang = c * np.log(a_load) - gammaln(c + 1) - np.log(1.0 - rho)
    max_log    = max(log_terms.max(), log_erlang)
    s = np.sum(np.exp(log_terms - max_log))
    e = np.exp(log_erlang - max_log)
    return float(e / (s + e))


def wq_mmc(lam, c, S_tx):
    """Fila M/M/c: retorna (W_q, rho, P_C)."""
    rho = lam * S_tx / c
    if rho >= 1.0:
        return np.inf, rho, 1.0
    P_C = erlang_c(int(c), rho)
    W_q = P_C * S_tx / (c * (1.0 - rho))
    return W_q, rho, P_C


In [ ]:
# ── Varredura N = 185 … 255 ────────────────────────────────
N_range  = np.arange(185, 256, dtype=int)
res_2cpu = []   # (W_q, rho, P_C, X_sql, S_tx, R_vec, U_vec)
res_1cpu = []

for n in N_range:
    ni = int(n)
    for two_cpu, lst in [(True, res_2cpu), (False, res_1cpu)]:
        X, R, U = solve_cs(ni, two_cpu=two_cpu)
        S_tx    = N_SQL * ni / X          # tempo de serviço no servidor de telmkt
        W, rho, PC = wq_mmc(lam_calls, ni, S_tx)
        lst.append((W, rho, PC, X, S_tx, R.copy(), U.copy()))

TARGET = 5.0   # SLA: W_q ≤ 5 s

def first_n_below(results, target):
    for i, r in enumerate(results):
        if np.isfinite(r[0]) and r[0] <= target:
            return int(N_range[i])
    return None

min_N_2cpu = first_n_below(res_2cpu, TARGET)
min_N_1cpu = first_n_below(res_1cpu, TARGET)

print(f"{'Configuração':<25} {'N mín':>7}  {'W_q (s)':>10}  {'ρ':>9}")
print("-" * 57)
for label, mn, results in [("2 processadores", min_N_2cpu, res_2cpu),
                             ("1 processador",  min_N_1cpu, res_1cpu)]:
    if mn is not None:
        idx = mn - int(N_range[0])
        W, rho, *_ = results[idx]
        print(f"{label:<25} {mn:>7}  {W:>10.3f}  {rho:>9.6f}")


In [ ]:
# ── Métricas detalhadas em min_N_2cpu ─────────────────────
if min_N_2cpu is not None:
    n   = int(min_N_2cpu)
    idx = n - int(N_range[0])
    W, rho, PC, X_sql, S_tx, R, U = res_2cpu[idx]
    X_tx = X_sql / N_SQL

    print(f"{'='*58}")
    print(f"  2 processadores  |  N = {n} representantes")
    print(f"{'='*58}")
    print(f"  X_sql = {X_sql:.5f} req/s     X_tx = {X_tx:.5f} tx/s")
    print(f"  S_tx  = {S_tx:.4f} s")
    print()
    print(f"  {'Estação':<10} {'Demanda (ms)':>12} {'Residência (ms)':>16} {'Util':>8}")
    print(f"  {'-'*52}")
    for nm, d, r, u in zip(['LAN','CPU','Disco'], [D_LAN,D_CPU,D_DSK], R, U):
        print(f"  {nm:<10} {d*1e3:>12.3f} {r*1e3:>16.3f} {u:>8.5f}")
    print()
    print(f"  M/M/{n} — Fila de Telemarketing")
    print(f"  λ = {lam_calls:.6f} chamadas/s")
    print(f"  ρ = {rho:.6f}   P_C = {PC:.6f}   W_q = {W:.4f} s")
    ok = "✓ W_q ≤ 5 s" if W <= 5.0 else "✗ EXCEDE 5 s"
    print(f"  [{ok}]")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Gráfico 1: W_q vs N ---
ax1 = axes[0]
Wq2 = [r[0] if np.isfinite(r[0]) and r[0] < 200 else np.nan for r in res_2cpu]
Wq1 = [r[0] if np.isfinite(r[0]) and r[0] < 200 else np.nan for r in res_1cpu]
ax1.plot(N_range, Wq2, 'b-o', ms=3, label='2 processadores')
ax1.plot(N_range, Wq1, 'r--s', ms=3, label='1 processador')
ax1.axhline(5.0, color='green', ls=':', lw=1.5, label='W_q = 5 s (SLA)')
for mn, col in [(min_N_2cpu, 'blue'), (min_N_1cpu, 'red')]:
    if mn is not None:
        ax1.axvline(mn, color=col, ls=':', lw=1, alpha=0.7)
        yoff = 12 if col == 'blue' else 18
        ax1.text(mn + 0.5, yoff, f'N={mn}', color=col, fontsize=9)
ax1.set_xlabel('N (representantes)')
ax1.set_ylabel('W_q (s)')
ax1.set_title('Tempo de Espera — Fila de Telemarketing')
ax1.set_ylim(0, 35)
ax1.legend()
ax1.grid(True, alpha=0.3)

# --- Gráfico 2: Utilizações (2-CPU) vs N ---
ax2 = axes[1]
ax2.plot(N_range, [r[6][0] for r in res_2cpu], '.-', ms=3, label='LAN')
ax2.plot(N_range, [r[6][1] for r in res_2cpu], '.-', ms=3, label='CPU (2-proc)')
ax2.plot(N_range, [r[6][2] for r in res_2cpu], '.-', ms=3, label='Disco')
ax2.plot(N_range, [r[1] if np.isfinite(r[0]) else np.nan for r in res_2cpu],
         '--', ms=3, label='ρ telemarketing')
ax2.axhline(1.0, color='black', ls=':', lw=1)
if min_N_2cpu is not None:
    ax2.axvline(min_N_2cpu, color='gray', ls=':', lw=1.5, alpha=0.8)
ax2.set_xlabel('N (representantes)')
ax2.set_ylabel('Utilização')
ax2.set_title('Utilizações vs N (2 processadores)')
ax2.set_ylim(0, 1.05)
ax2.legend(fontsize=8)
ax2.grid(True, alpha=0.3)

plt.suptitle('Exercício 7.1 — Modelo CS com 2 processadores', fontsize=12)
plt.tight_layout()
plt.savefig('/home/renan/workspace/ads/cap7_ex71.png', dpi=100, bbox_inches='tight')
plt.show()
print('Figura salva em cap7_ex71.png')
